# Project tour

This notebook is generated by `tools/build_notebooks.py`. Edit the builder, not this file: a test fails if the two drift apart.

It walks through the pieces of the repo that exist at milestone 0, so later notebooks can import them without surprises. Every code cell calls the real package, `arima_forecasting`. Nothing is copied from `src/`.

## Repo layout

| Path | What lives there |
|---|---|
| `src/arima_forecasting/` | All the logic: config, the JSON contract, provenance. Tested code that notebooks import. |
| `tests/` | pytest tests. Expected values are worked out by hand, with the arithmetic in comments. |
| `tools/` | Thin command-line wrappers: build the notebooks, export JSON for the app. |
| `notebooks/` | Generated notebooks, this one included. |
| `app/` | The small web app that reads the exported JSON. |
| `docs/` | Decisions (ADRs), architecture notes and concept explainers. |
| `reports/` | Written results. |

The rule: logic lives in `src/`, tools only wrap it, notebooks only call it. That keeps every number reproducible and testable.

In [ ]:
import arima_forecasting
from arima_forecasting import config, contract, export

print("arima_forecasting", arima_forecasting.__version__)

## Where the data lives

Raw market data is never committed. It lives outside the repo, under a shared data root:

- the environment variable `VOL_DATA_DIR`, if it is set and not empty;
- otherwise `~/data/vol-projects`.

This project's raw files go in `<data root>/arima-stock-forecasting/raw`. `raw_dir()` only builds the path. It never creates a folder, so calling it is safe anywhere.

In [ ]:
print("data root:", config.data_root())
print("raw dir:  ", config.raw_dir())

# Both functions take an optional `env` mapping, so we can see the override
# without touching the real environment.
print("with override:", config.raw_dir({"VOL_DATA_DIR": "/mnt/market-data"}))

## Two study windows

| Window | Start | End (exclusive) | Purpose |
|---|---|---|---|
| `course` | 2015-01-01 | 2024-01-01 | The period the course outline uses. We fit and choose models here. |
| `oos` | 2024-01-01 | 2026-10-01 | A frozen out-of-sample period. We only score forecasts here. |

"End (exclusive)" means the end date itself is not in the window. This matches yfinance, whose `end` argument is exclusive. The course outline asks for `end=2023-12-31`. That day is excluded, and the 30th and 31st are a weekend anyway, so the last bar is Friday 2023-12-29. Writing the end as 2024-01-01 says the same thing, and it makes the two windows touch with no gap and no overlap.

The out-of-sample window is frozen so that anyone who re-runs the study gets the same numbers.

In [ ]:
for window in (config.COURSE_WINDOW, config.OOS_WINDOW):
    days = (window.end_exclusive - window.start).days
    print(f"{window.name:>6}: {window.start} -> {window.end_exclusive} (exclusive), {days} calendar days")

# The windows tile: one ends exactly where the next begins.
assert config.COURSE_WINDOW.end_exclusive == config.OOS_WINDOW.start

# A window that ends before it starts is a bug, so it refuses to exist.
try:
    config.StudyWindow("backwards", config.OOS_WINDOW.end_exclusive, config.OOS_WINDOW.start)
except ValueError as error:
    print("rejected:", error)

## The JSON envelope

Python writes JSON files that the web app reads. Every file has the same outer shape, an *envelope*, so the app can check what it received before it draws anything:

| Key | Meaning |
|---|---|
| `schema_version` | Semver. A new major version is a breaking change, and the app refuses a major it does not know. |
| `artifact` | Which payload this is, for example `meta`. |
| `generated_at` | UTC time, written `YYYY-MM-DDTHH:MM:SSZ`. |
| `source_commit` | The git commit that produced the file. `-dirty` is added if the working tree had uncommitted changes, and `uncommitted` means there was no commit yet. |
| `data` | The payload itself. |

`make_envelope` builds one and refuses bad input. `validate_envelope` lists every problem it finds. `dumps` writes the text, and it refuses NaN and Infinity: they are not valid JSON, and the browser's `JSON.parse` would reject the whole file.

In [ ]:
from datetime import datetime, timedelta, timezone

# 12:00 in a UTC+2 timezone is 10:00 UTC. The envelope always stores UTC.
local_noon = datetime(2026, 10, 28, 12, 0, tzinfo=timezone(timedelta(hours=2)))
envelope = contract.make_envelope(
    "example",
    {"ticker": config.TICKER, "rows": 3},
    generated_at=local_noon,
    source_commit="uncommitted",
)
print(contract.dumps(envelope))
print("problems:", contract.validate_envelope(envelope))

In [ ]:
# NaN is not valid JSON, so dumps raises instead of writing a file the app cannot read.
broken = contract.make_envelope(
    "example",
    {"value": float("nan")},
    generated_at=local_noon,
    source_commit="uncommitted",
)
try:
    contract.dumps(broken)
except ValueError as error:
    print("rejected:", error)

## What comes next

The milestones below are the same list the app shows. Each one adds code to `src/arima_forecasting/`, tests for it, and a notebook that imports it.

In [ ]:
for milestone in export.meta_payload()["milestones"]:
    print(f"{milestone['id']}  {milestone['status']:<8}  {milestone['title']}")